In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score,f1_score,recall_score,roc_auc_score,precision_score
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV

import joblib

In [3]:
df=pd.read_csv('/content/phishing_processed.csv')

In [4]:
df.shape

(11190, 16)

In [5]:
df.head()

,domain_age,phish_hints,ratio_extHyperlinks,ratio_intHyperlinks,safe_anchor,ratio_extRedirection,length_url,ratio_digits_url,length_words_raw,longest_words_raw,length_hostname,avg_word_path,domain_in_title,char_repeat,status,status-encoded
0,5075.0,0,0.470588,0.529412,0.0,0.875000,37,0.000000,4,11,19,4.500000,0,4,legitimate,0
1,5767.0,0,0.033333,0.966667,100.0,0.000000,77,0.220779,4,32,23,14.666667,1,4,phishing,1
2,4004.0,0,0.000000,1.000000,100.0,0.000000,126,0.150794,12,17,50,8.142857,1,2,phishing,1
3,5075.0,0,0.026846,0.973154,62.5,0.250000,18,0.000000,1,5,11,0.000000,1,0,legitimate,0
4,8175.0,0,0.529412,0.470588,0.0,0.537037,55,0.000000,6,11,15,7.000000,0,3,legitimate,0


In [8]:
X=df.iloc[:,0:-2]
y=df.iloc[:,-1]

In [9]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [12]:
print(X_train.shape)
print(X_test.shape)

(8952, 14)
(2238, 14)


In [13]:
rf=RandomForestClassifier(n_estimators=400,max_features=0.75,max_samples=0.5,random_state=42,n_jobs=-1)



In [14]:
rf.fit(X_train,y_train)

RandomForestClassifier(max_features=0.75, max_samples=0.5, n_estimators=400,
                       n_jobs=-1, random_state=42)

In [88]:
y_pred=rf.predict(X_test)

In [17]:
print("Accuracy :",accuracy_score(y_test,y_pred))
print("Precision:",precision_score(y_test,y_pred))
print("Recall:",recall_score(y_test,y_pred))
print("F1 score:",f1_score(y_test,y_pred))

Accuracy : 0.920017873100983
Precision: 0.9051959890610757
Recall: 0.9297752808988764
F1 score: 0.9173210161662817


- so accuracy is 92% means 92% prediction is correct

- precision is 90% means the out of all websites predicted as phishing , 90% were phishing

- Recall is 92% means the out of actual phishing url how many it catches is 92%

- F1 score is combination of precision and recall which is 91%


### Classification-report

In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      1170
           1       0.91      0.93      0.92      1068

    accuracy                           0.92      2238
   macro avg       0.92      0.92      0.92      2238
weighted avg       0.92      0.92      0.92      2238



### Hyperparameter Tuning
#### (RandomSearchCV)

In [71]:
n_estimators=[20,60,100,200]
max_features=[0.25,0.5,0.75]
max_samples=[0.2,0.5,0.75]
max_depth=[2,8,None]

min_samples_leaf=[1,2]
min_samples_split=[2,5]


In [72]:
params={
    'n_estimators':n_estimators,
    'max_features':max_features,
    'max_samples':max_samples,
    'max_depth':max_depth,
    'min_samples_split':min_samples_split,
    'min_samples_leaf':min_samples_leaf
}

In [91]:
rf_random_search=RandomizedSearchCV(estimator=rf,param_distributions=params,cv=5,verbose=2,n_jobs=-1)
rf_random_search.fit(X_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(max_features=0.75,
                                                    max_samples=0.5,
                                                    n_estimators=400, n_jobs=-1,
                                                    random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [2, 8, None],
                                        'max_features': [0.25, 0.5, 0.75],
                                        'max_samples': [0.2, 0.5, 0.75],
                                        'min_samples_leaf': [1, 2],
                                        'min_samples_split': [2, 5],
                                        'n_estimators': [20, 60, 100, 200]},
                   verbose=2)

In [92]:
rf_random_search.best_params_

{'n_estimators': 100,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_samples': 0.5,
 'max_features': 0.5,
 'max_depth': None}

In [93]:
rf_random_search.best_score_

np.float64(0.9138736513105565)

In [94]:
best_rf=rf_random_search.best_estimator_

##Saving best model

In [44]:
joblib.dump(
    best_rf,
    "random_forest_phishing.pkl"
)

['random_forest_phishing.pkl']

In [84]:
rf_best = joblib.load("random_forest_phishing.pkl")

In [85]:
y_pred = rf_best.predict(X_test)


In [86]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))


Accuracy : 0.920017873100983
Precision: 0.9029918404351768
Recall   : 0.9325842696629213
F1 Score : 0.9175495163519115


- The baseline Random Forest model and the hyperparameter-tuned Random Forest model achieved very similar performance.

- The tuned model obtained a slightly higher recall and F1-score, while the baseline model achieved marginally higher precision.


- Since phishing detection prioritizes minimizing false negatives and maximizing phishing detection rates, the tuned Random Forest model obtained through RandomizedSearchCV was selected as the final model for deployment.